Here we will finally work with all of the cleaned datasets:

1. The merged DrugBank DTI dataset and Yamanishi DTI dataset
2. Negative DTI dataset

We will do two things:
 1. We will add labels to 2. (the positive is already labeled)
 2. We will do feature extraction

In [ ]:
#get all the datasets

import pandas as pd
from pathlib import Path


DIR_POSITIVE_DRUGBANK = Path("../data/positive dti datasets/drugbank/drugbank final dataset")
DIR_POSITIVE_YAMANISHI = Path("../data/positive dti datasets/yamanishi/final yamanishi")

DIR_NEGATIVE = Path("../data/negative dti datasets/negative_datasets_final_merged")


df_positive_drugbank = pd.read_csv(DIR_POSITIVE_DRUGBANK/"drugbank_filtered_final.csv")
df_positive_yamanishi = pd.read_csv(DIR_POSITIVE_YAMANISHI/"yamanishi_final.csv")


csv_path = DIR_NEGATIVE / "negative_final.csv"
df_negative = pd.read_csv(csv_path)

#print("Positive Drugbank DTI dataset head:", df_positive_drugbank.head)
#print("Positive Yamanishi DTI dataset head:", df_positive_yamanishi.head)
#print("Negative DTI dataset head:", df_negative.head)




In [ ]:
#Add labels to yamanishi and negative datasets
df_positive_yamanishi['label'] = 1
df_negative['label'] = 0


print("Yamanishi DTI dataset with labels head:", df_positive_yamanishi.shape)
print("Negative DTI dataset with labels head:", df_negative.shape)
print("DrugBank DTI dataset with labels head:", df_positive_drugbank.shape)

In [ ]:
# drop duplicates before feature extraction
df_positive_drugbank['pair_id'] = df_positive_drugbank['smiles'] + "_" + df_positive_drugbank['protein_sequence']
df_positive_drugbank = df_positive_drugbank.drop_duplicates(subset=['pair_id'])


In [ ]:
print("Drugbank after dropping duplicates:", df_positive_drugbank.shape)

In [ ]:
# Feature extraction functions

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem


# Morgan fingerprint
def smiles_to_morgan(smiles, radius=2, n_bits=1024):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: #invalid SMILES
        return np.zeros(n_bits) #we fill the fingerprint with zeros
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits) #1024-bit fingerprint
    arr = np.zeros((n_bits,), dtype=int)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr) #convert fingerprint to numpy array
    return arr

# Amino Acid Composition
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

def protein_aac(seq):
    seq = ''.join(filter(str.isalpha, seq.upper())) #clean sequence
    length = len(seq) #length of sequence
    return np.array([seq.count(aa)/length if length > 0 else 0 for aa in amino_acids]) #AAC vector




In [ ]:
import requests
from tqdm import tqdm

#Helper functions to get sequences from uniprot for yamanishi and negative datasets

# Takes a list of UniProt IDs and retrieves their corresponding sequences in FASTA format 
# It returns a dictionary mapping each UniProt ID to its protein sequence as a single string

def fetch_uniprot_sequences_bulk(uniprot_ids):
    seqs = {}

    for uid in tqdm(uniprot_ids):
        try:
            # clean possible prefixes
            if uid.startswith("up:"):
                uid = uid.replace("up:", "")

            url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
            r = requests.get(url, timeout=10)

            if r.status_code != 200:
                seqs[uid] = None
                continue

            seq = "".join(r.text.split("\n")[1:]).strip()
            seqs[uid] = seq
        except:
            seqs[uid] = None

    return seqs


In [ ]:
###=========== HELPER FUNCTIONS TO GET SMILES FROM KEGG IDS yamanishi ===========###


import requests
def kegg_to_pubchem(kegg_id):
    url = f"http://rest.kegg.jp/conv/pubchem/{kegg_id}"
    try:
        r = requests.get(url, timeout=10)   # ⬅ 10-second timeout
        if r.status_code != 200 or not r.text.strip():
            return None
        return r.text.strip().split("\t")[1].replace("pubchem:", "")
    except:
        return None
def pubchem_to_smiles(cid):
    if cid is None:
        return None

    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/CanonicalSMILES/TXT"
    r = requests.get(url, timeout=10)
    
    if r.status_code != 200 or len(r.text.strip()) == 0:
        return None
    
    return r.text.strip()

In [69]:
# check for duplicate pairs in yamanishi dataset use pair column
df_positive_yamanishi['pair'].nunique()
df_positive_yamanishi.shape
# shape and number unique pairs is the same so no duplicates

(5127, 5)

In [ ]:
#We are gonna get the protein sequences for yamanishi and the smiles
# and then we are gonna merge yamanishi with drug bank and take care of duplicates

In [70]:

unique_uniprots = df_positive_yamanishi["uniprot_id"].dropna().unique()
uniprot_to_seq = fetch_uniprot_sequences_bulk(unique_uniprots)

100%|██████████| 987/987 [04:46<00:00,  3.45it/s]


In [72]:

# map back to dataframe
df_positive_yamanishi["protein_sequence"] = (
    df_positive_yamanishi["uniprot_id"].map(uniprot_to_seq)
)

In [73]:
df_positive_yamanishi["protein_sequence"].isna().sum()


3

In [74]:
df_positive_yamanishi_AFTER = df_positive_yamanishi.dropna(subset=["protein_sequence"])

In [75]:
df_positive_yamanishi_AFTER["protein_sequence"].isna().sum()


0

In [76]:
unique_drugs = df_positive_yamanishi["kegg_drug_id"].dropna().unique()

mapping_kegg_to_pubchem = {
    d: kegg_to_pubchem(d) for d in tqdm(unique_drugs)
}

df_positive_yamanishi["pubchem_cid"] = df_positive_yamanishi["kegg_drug_id"].map(mapping_kegg_to_pubchem)



100%|██████████| 791/791 [24:06<00:00,  1.83s/it]


In [77]:
unique_cids = df_positive_yamanishi["pubchem_cid"].dropna().unique()
len(unique_cids)


791

In [78]:
cid_to_smiles = {}

for cid in tqdm(unique_cids):
    cid_to_smiles[cid] = pubchem_to_smiles(cid)

100%|██████████| 791/791 [06:46<00:00,  1.95it/s]


In [79]:
df_positive_yamanishi["smiles"] = df_positive_yamanishi["pubchem_cid"].map(cid_to_smiles)


In [80]:
df_positive_yamanishi["smiles"].shape

(5127,)

In [81]:
(df_positive_yamanishi["protein_sequence"].isna().sum(),
 (df_positive_yamanishi["protein_sequence"] == None).sum(),
 df_positive_yamanishi["protein_sequence"].apply(lambda x: x is None).sum()
)

(3, 0, 0)

In [82]:
df_positive_yamanishi = df_positive_yamanishi[
    df_positive_yamanishi["protein_sequence"].notna() &
    df_positive_yamanishi["protein_sequence"].apply(lambda x: isinstance(x, str))
].copy()


In [83]:
#see how many smiles are NaN
df_positive_yamanishi["smiles"].isna().sum()
#DROP NaN smiles
df_positive_yamanishi = df_positive_yamanishi[
    df_positive_yamanishi["smiles"].notna() &
    df_positive_yamanishi["smiles"].apply(lambda x: isinstance(x, str))
].copy()

In [84]:
print("yam columns :", df_positive_yamanishi.columns)
print("db columns:", df_positive_drugbank.columns)

yam columns : Index(['kegg_protein_id', 'kegg_drug_id', 'dataset', 'uniprot_id', 'pair',
       'protein_sequence', 'pubchem_cid', 'smiles'],
      dtype='object')
db columns: Index(['drugbank_id', 'drug_name', 'smiles', 'kegg_drug_id', 'uniprot_id',
       'kegg_protein_id', 'protein_sequence', 'organism', 'label', 'pair_id'],
      dtype='object')


In [115]:
df_positive_drugbank["protein_sequence"] = df_positive_drugbank["protein_sequence"].str.replace(">", "", regex=False)

In [116]:
# DrugBank positives
df_db_min = df_positive_drugbank[[
    "protein_sequence",
    "smiles"
]].copy()

# Yamanishi positives
df_yam_min = df_positive_yamanishi[[
    "protein_sequence",
    "smiles"
]].copy()


In [118]:
def clean_df(df):
    return df[
        df["protein_sequence"].notna() &
        df["smiles"].notna() &
        df["protein_sequence"].apply(lambda x: isinstance(x, str)) &
        df["smiles"].apply(lambda x: isinstance(x, str))
    ].copy()

df_db_min = clean_df(df_db_min)
df_yam_min = clean_df(df_yam_min)


In [119]:
df_pos_all = pd.concat(
    [df_db_min, df_yam_min],
    axis=0,
    ignore_index=True
)


In [120]:
df_pos_all.shape

(28852, 2)

In [121]:
n_dupes = df_pos_all.duplicated(
    subset=["smiles", "protein_sequence"]
).sum()

print("Number of duplicated DTI pairs:", n_dupes)


Number of duplicated DTI pairs: 63


In [122]:
df_pos_all = df_pos_all.drop_duplicates(
    subset=["smiles", "protein_sequence"]
).reset_index(drop=True)


In [123]:
df_pos_all.shape

(28789, 2)

In [124]:
print(
    "Duplicate DTI pairs:",
    df_pos_all.duplicated(subset=["smiles", "protein_sequence"]).sum()
)

Duplicate DTI pairs: 0


In [125]:
#save this df_pos_all as a csv
df_pos_all.to_csv("../data/positive dti datasets/combined_positive_dti_dataset.csv", index=False)

In [130]:
df_pos_all.shape

(28789, 3)

In [129]:
# add label column to df_pos_all thats all 1s
df_pos_all['label'] = 1

In [131]:

###### BEFORE THIS WE ARE GONNA TAKE CARE OF YAMANISHI DATASET FIRST ######

#Feature extraction for Positives

#smiles 
#protein_sequence 
#label 

# Clean sequences

# Combine features
X_pos, y_pos = [], df_pos_all["label"].values

#X = features/input variables (what the model learns from)
#y = labels/target variables (what the model predicts)

# X is filled with the combined Morgan fingerprints and Amino Acid Composition from the protein sequences

for _, row in df_pos_all.iterrows():
    x = np.concatenate([
        smiles_to_morgan(row["smiles"]),
        protein_aac(row["protein_sequence"])
    ])
    X_pos.append(x)

X_pos = np.array(X_pos)

[15:26:03] Unusual charge on atom 0 number of radical electrons set to zero


In [134]:
#see how many bad molecules are in the dataset
def check_bad_molecules(df, smiles_column):
    bad = 0
    total = len(df_positive_drugbank)

    for s in df_positive_drugbank["smiles"]:
        if Chem.MolFromSmiles(s) is None:
            bad += 1

    print(f"Bad molecules: {bad}/{total}")
   


In [136]:
df_pos_all.shape

(28789, 3)

In [135]:
check_bad_molecules(df_pos_all, "smiles")

[15:27:16] Unusual charge on atom 0 number of radical electrons set to zero


Bad molecules: 0/23728


In [98]:
df_negative.columns
# rename UniProt (SwissProt) Primary ID of Target Chain 1 to UniProtIds
df_negative = df_negative.rename(columns={"UniProt (SwissProt) Primary ID of Target Chain 1": "UniProtIds"})  
df_negative = df_negative.rename(columns={"Ligand SMILES": "smiles"})  


In [99]:
df_negative.columns

Index(['smiles', 'UniProtIds', 'Affinity', 'pair_id'], dtype='object')

In [100]:
# Drop NaN in Terget column. 
print("Negative dataset shape before dropping NaN UniProtIds", df_negative.shape)
df_negative = df_negative[df_negative["UniProtIds"].notna()]
print("After dropping NaN:", df_negative.shape)

Negative dataset shape before dropping NaN UniProtIds (225462, 4)
After dropping NaN: (225462, 4)


In [101]:


print("Negative dataset shape before dropping NaN for Drug", df_negative.shape)
df_negative = df_negative[df_negative["smiles"].notna()]
print("After dropping NaN for Drug:", df_negative.shape)


Negative dataset shape before dropping NaN for Drug (225462, 4)
After dropping NaN for Drug: (225462, 4)


In [102]:
#check how many unique uniprot ids are in the negative dataset
df_negative['UniProtIds'].nunique()


4512

In [103]:
unique_uniprots = df_negative['UniProtIds'].unique()

In [104]:
# check for duplicate pairs in negative dataset
pair_ids = df_negative['smiles'] + "_" + df_negative['UniProtIds']
duplicates = pair_ids[pair_ids.duplicated()]
print(f"Number of duplicate pairs in negative dataset: {len(duplicates)}")

Number of duplicate pairs in negative dataset: 0


In [105]:
# Fetch sequences once
seq_map = fetch_uniprot_sequences_bulk(unique_uniprots)

100%|██████████| 4512/4512 [22:38<00:00,  3.32it/s]  


In [106]:
# Attach to main dataframe
df_negative["protein_sequence"] = df_negative["UniProtIds"].map(seq_map)

In [137]:
df_negative["protein_sequence"].isna().sum()

35

In [108]:
df_negative["protein_sequence"].notna().sum()

225427

In [109]:
df_negative_AFTER = df_negative.dropna(subset=["protein_sequence"])
df_negative_AFTER["protein_sequence"].isna().sum()

0

In [110]:
df_negative_AFTER.shape

(225427, 5)

In [111]:
df_negative_AFTER.head()

,smiles,UniProtIds,Affinity,pair_id,protein_sequence
0,CC(=O)NCCN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](O)[C...,P03367,12500.0,CC(=O)NCCN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](O)[C...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...
1,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCNC(=O)Cc...,P03367,12500.0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCNC(=O)Cc...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...
2,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCn2ccn...,P03367,12500.0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCn2ccn...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...
3,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCN2CCO...,P03367,12500.0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCN2CCO...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...
4,CC(C)(C)C(O)CN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](...,P03367,21000.0,CC(C)(C)C(O)CN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...


In [139]:
df_negative_AFTER['label'] = 0

/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_67433/1437344307.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_negative_AFTER['label'] = 0


In [140]:
df_negative_AFTER

,smiles,UniProtIds,Affinity,pair_id,protein_sequence,label
0,CC(=O)NCCN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](O)[C...,P03367,12500.0,CC(=O)NCCN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](O)[C...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...,0
1,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCNC(=O)Cc...,P03367,12500.0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCNC(=O)Cc...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...,0
2,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCn2ccn...,P03367,12500.0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCn2ccn...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...,0
3,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCN2CCO...,P03367,12500.0,O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCN2CCO...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...,0
4,CC(C)(C)C(O)CN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](...,P03367,21000.0,CC(C)(C)C(O)CN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](...,MGARASVLSGGELDRWEKIRLRPGGKKKYKLKHIVWASRELERFAV...,0
...,...,...,...,...,...,...
225458,CN(C)CCCNc1nc(NCC2CCNCC2)c2cc(ccc2n1)-c1ccccc1,Q8NG06,24000.0,CN(C)CCCNc1nc(NCC2CCNCC2)c2cc(ccc2n1)-c1ccccc1...,MAWAPPGERLREDARCPVCLDFLQEPVSVDCGHSFCLRCISEFCEK...,0
225459,C[C@H]1CN(Cc2ccccc2C(=O)N[C@H]2N=C(c3ccccc3)c3...,Q13191,37200.0,C[C@H]1CN(Cc2ccccc2C(=O)N[C@H]2N=C(c3ccccc3)c3...,MANSMNGRNPGGRGGNPRKGRILGIIDAIQDAVGPPKQAAADRRTV...,0
225460,O=C(NC1N=C(c2cc[nH]c(=O)c2)c2ccccc2NC1=O)c1ccc...,Q13191,11000.0,O=C(NC1N=C(c2cc[nH]c(=O)c2)c2ccccc2NC1=O)c1ccc...,MANSMNGRNPGGRGGNPRKGRILGIIDAIQDAVGPPKQAAADRRTV...,0
225461,O=C(NC1N=C(c2ccccc2NC1=O)c1ccc[nH]c1=O)c1ccccc...,Q13191,20000.0,O=C(NC1N=C(c2ccccc2NC1=O)c1ccc[nH]c1=O)c1ccccc...,MANSMNGRNPGGRGGNPRKGRILGIIDAIQDAVGPPKQAAADRRTV...,0


In [141]:
X_neg, y_neg = [], df_negative_AFTER["label"].values

for _, row in df_negative_AFTER.iterrows():
    x = np.concatenate([
        smiles_to_morgan(row["smiles"]),
        protein_aac(row["protein_sequence"])
    ])
    X_neg.append(x)

X_neg = np.array(X_neg)


[15:30:14] Explicit valence for atom # 19 N, 4, is greater than permitted
[15:30:14] Explicit valence for atom # 19 N, 4, is greater than permitted
[15:30:14] Explicit valence for atom # 4 N, 4, is greater than permitted
[15:30:15] Explicit valence for atom # 13 N, 4, is greater than permitted
[15:30:15] Explicit valence for atom # 17 N, 4, is greater than permitted
[15:30:17] Can't kekulize mol.  Unkekulized atoms: 7 8 9 10 11 12 13 14 15
[15:30:17] Explicit valence for atom # 22 N, 4, is greater than permitted
[15:30:18] Explicit valence for atom # 17 N, 4, is greater than permitted
[15:30:18] Can't kekulize mol.  Unkekulized atoms: 1 2 3 5 35
[15:30:18] Can't kekulize mol.  Unkekulized atoms: 16 17 19 20 22
[15:30:18] Can't kekulize mol.  Unkekulized atoms: 26 27 28 29 31
[15:30:18] Can't kekulize mol.  Unkekulized atoms: 15 16 18 19 21
[15:30:18] Can't kekulize mol.  Unkekulized atoms: 25 26 28 29 31
[15:30:18] Can't kekulize mol.  Unkekulized atoms: 15 16 18 19 21
[15:30:18] Can't

In [142]:
X_neg.shape, y_neg.shape

((225427, 1044), (225427,))

In [143]:
X_pos.shape, y_pos.shape

((28789, 1044), (28789,))

In [153]:
from sklearn.model_selection import train_test_split

# Split positives
X_pos_train, X_pos_test, y_pos_train, y_pos_test = train_test_split(
    X_pos, y_pos, test_size=0.2, random_state=42
)

# Split negatives
X_neg_train, X_neg_test, y_neg_train, y_neg_test = train_test_split(
    X_neg, y_neg, test_size=0.2, random_state=42
)


In [154]:

import numpy as np
from sklearn.utils import shuffle

# Downsample negatives in train
n_pos_train = len(X_pos_train)
X_neg_train_down = X_neg_train[np.random.choice(len(X_neg_train), n_pos_train, replace=False)]
y_neg_train_down = y_neg_train[np.random.choice(len(y_neg_train), n_pos_train, replace=False)]

# Combine positives and negatives
X_train_bal = np.vstack([X_pos_train, X_neg_train_down])
y_train_bal = np.concatenate([y_pos_train.ravel(), y_neg_train_down.ravel()])

# Shuffle
X_train_bal, y_train_bal = shuffle(X_train_bal, y_train_bal, random_state=42)


In [155]:
# Combine positives and negatives for test
X_test = np.vstack([X_pos_test, X_neg_test])
y_test = np.concatenate([y_pos_test.ravel(), y_neg_test.ravel()])

# Shuffle
X_test, y_test = shuffle(X_test, y_test, random_state=42)


In [156]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
)
xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=600,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [157]:
from sklearn.metrics import roc_auc_score, classification_report

y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]  # probability for positive class

# Metrics
print("AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))


AUC: 0.9978783436331018
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     45086
           1       0.99      0.88      0.93      5758

    accuracy                           0.99     50844
   macro avg       0.99      0.94      0.96     50844
weighted avg       0.99      0.99      0.99     50844



Generate a specialized test set 

From the top 10 ranked proteins, they created an all-against-all DTI test set using all 3,081 unique ligands from training/test data.

In [ ]:
import pandas as pd
import numpy as np

# taken top proteins from the original study
top_proteins_uniprot = [
    "Q5HDY4",
    "Q5HID0",
    "Q5HID3",
    "Q5HCP4",
    "Q5HGF8",
    "Q5HIC8",
    "Q5HDW3",
    "Q5HDW4",
    "Q5HDZ0",
    "Q5HIH4"
]
#fetch_uniprot_sequences_bulk

top_protein_sequences = fetch_uniprot_sequences_bulk(top_proteins_uniprot)



100%|██████████| 10/10 [00:02<00:00,  4.79it/s]


In [171]:
top_protein_sequences

{'Q5HDY4': 'MIEIEKPRIETIEISEDAKFGKFVVEPLERGYGTTLGNSLRRILLSSLPGAAVKYIEIEGVLHEFSAVDNVVEDVSTIIMNIKQLALKIYSEEDKTLEIDVRDEGEVTASDITHDSDVEILNPELKIATVSKGGHLKIRLVANKGRGYALAEQNNTSDLPIGVIPVDSLYSPVERVNYTVENTRVGQSSDFDKLTLDVWTNGSITPQESVSLAAKIMTEHLNIFVGLTDEAQNAEIMIEKEEDQKEKVLEMSIEELDLSVRSYNCLKRAGINSVQELADKSEADMMKVRNLGRKSLEEVKYKLEDLGLGLRKED',
 'Q5HID0': 'MPTINQLVRKPRQSKIKKSDSPALNKGFNSKKKKFTDLNSPQKRGVCTRVGTMTPRKPNSALRKYARVRLSNNIEINAYIPGIGHNLQEHSVVLVRGGRVRDLPGVRYHIVRGALDTSGVDGRRQGRSLYGTKKPKN',
 'Q5HID3': 'MAGQVVQYGRHRKRRNYARISEVLELPNLIEIQTKSYEWFLREGLIEMFRDISPIEDFTGNLSLEFVDYRLGEPKYDLEESKNRDATYAAPLRVKVRLIIKETGEVKEQEVFMGDFPLMTDTGTFVINGAERVIVSQLVRSPSVYFNEKIDKNGRENYDATIIPNRGAWLEYETDAKDVVYVRIDRTRKLPLTVLLRALGFSSDQEIVDLLGDNEYLRNTLEKDGTENTEQALLEIYERLRPGEPPTVENAKSLLYSRFFDPKRYDLASVGRYKTNKKLHLKHRLFNQKLAEPIVNTETGEIVVEEGTVLDRRKIDEIMDVLESNANSEVFELHGSVIDEPVEIQSIKVYVPNDDEGRTTTVIGNAFPDSEVKCITPADIIASMSYFFNLLSGIGYTDDIDHLGNRRLRSVGELLQNQFRIGLSRMERVVRERMSIQDTESITPQQLINIRPVIASIKEFFGSSQLSQFMDQANPLAELTHKRRLSALGPGGLTRERAQMEVRDVHY

In [ ]:
top_protein_sequences = list(top_protein_sequences.values())



In [174]:
print(len(top_protein_sequences))

10


In [175]:
ligands = df_pos_all["smiles"].unique().tolist()  # all unique ligands


In [176]:
len(ligands)

9120

In [177]:


# Create all possible pairs (protein x ligand)
test_pairs = []
for prot in top_protein_sequences:
    for lig in ligands:
        test_pairs.append((lig, prot))

print("Total DTI pairs:", len(test_pairs))


Total DTI pairs: 91200


In [164]:
len(test_pairs) 


91200

In [178]:
test_pairs[1]

('CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H](CC1=CC=C(O)C=C1)NC(=O)[C@H](CO)NC(=O)[C@H](CC1=CNC2=CC=CC=C12)NC(=O)[C@H](CC1=CN=CN1)NC(=O)[C@@H]1CCC(=O)N1)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)NNC(N)=O',
 'MIEIEKPRIETIEISEDAKFGKFVVEPLERGYGTTLGNSLRRILLSSLPGAAVKYIEIEGVLHEFSAVDNVVEDVSTIIMNIKQLALKIYSEEDKTLEIDVRDEGEVTASDITHDSDVEILNPELKIATVSKGGHLKIRLVANKGRGYALAEQNNTSDLPIGVIPVDSLYSPVERVNYTVENTRVGQSSDFDKLTLDVWTNGSITPQESVSLAAKIMTEHLNIFVGLTDEAQNAEIMIEKEEDQKEKVLEMSIEELDLSVRSYNCLKRAGINSVQELADKSEADMMKVRNLGRKSLEEVKYKLEDLGLGLRKED')

In [179]:

# Generate features
X_test_special = []
for lig, prot in test_pairs:
    feat = np.concatenate([
        smiles_to_morgan(lig),
        protein_aac(prot)
    ])
    X_test_special.append(feat)

X_test_special = np.array(X_test_special)


[16:22:52] Unusual charge on atom 0 number of radical electrons set to zero
[16:22:54] Unusual charge on atom 0 number of radical electrons set to zero
[16:22:55] Unusual charge on atom 0 number of radical electrons set to zero
[16:22:56] Unusual charge on atom 0 number of radical electrons set to zero
[16:22:57] Unusual charge on atom 0 number of radical electrons set to zero
[16:22:58] Unusual charge on atom 0 number of radical electrons set to zero
[16:23:00] Unusual charge on atom 0 number of radical electrons set to zero
[16:23:01] Unusual charge on atom 0 number of radical electrons set to zero
[16:23:02] Unusual charge on atom 0 number of radical electrons set to zero
[16:23:03] Unusual charge on atom 0 number of radical electrons set to zero


In [180]:


y_pred_proba = xgb.predict_proba(X_test_special)[:, 1]  # probability of interaction

# Combine with DTI info
df_test = pd.DataFrame(test_pairs, columns=["ligand_smiles", "protein_sequence"])
df_test["predicted_prob"] = y_pred_proba

# Sort by probability
df_test_sorted = df_test.sort_values("predicted_prob", ascending=False)

# Take top 5 most likely DTIs
top5 = df_test_sorted.head(5)
print(top5)


                                           ligand_smiles  \
63741  CC(C(=O)NC1=CC=C(C=C1)C(=O)N)OC(=O)COC2=CC=CC=...   
63184      CC(C(=O)NC1=CC=C(C=C1)F)OC(=O)COC2=CC=CC=C2Cl   
63317  CC(C(=O)NC1=CC=C(C=C1)C(=O)C)OC(=O)COC2=CC=CC=...   
63174             CC(C(=O)NC(=O)NC)OC(=O)COC1=CC=CC=C1Cl   
63118   CC(C(=O)NC1CCCC2=CC=CC=C12)OC(=O)COC3=CC=CC=C3Cl   

                                        protein_sequence  predicted_prob  
63741  MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...        0.999548  
63184  MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...        0.999464  
63317  MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...        0.999463  
63174  MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...        0.999258  
63118  MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...        0.999235  


In [ ]:
top5.drop

,ligand_smiles,protein_sequence,predicted_prob,UniProt ID,Protein Name
63741,CC(C(=O)NC1=CC=C(C=C1)C(=O)N)OC(=O)COC2=CC=CC=...,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999548,NaN,NaN
63184,CC(C(=O)NC1=CC=C(C=C1)F)OC(=O)COC2=CC=CC=C2Cl,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999464,NaN,NaN
63317,CC(C(=O)NC1=CC=C(C=C1)C(=O)C)OC(=O)COC2=CC=CC=...,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999463,NaN,NaN
63174,CC(C(=O)NC(=O)NC)OC(=O)COC1=CC=CC=C1Cl,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999258,NaN,NaN
63118,CC(C(=O)NC1CCCC2=CC=CC=C12)OC(=O)COC3=CC=CC=C3Cl,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999235,NaN,NaN


In [197]:
import requests

def smiles_to_pubchem_info(smiles):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles}/cids/TXT"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code != 200 or not r.text.strip():
            return None, None
        cid = r.text.strip()
        
        # Get compound info (title)
        info_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/IUPACName/JSON"
        r2 = requests.get(info_url, timeout=10).json()
        name = r2["PropertyTable"]["Properties"][0]["IUPACName"]
        
        return cid, name
    except:
        return None, None



In [198]:

top5["PubChem CID"], top5["Drug Name"] = zip(*top5["ligand_smiles"].map(smiles_to_pubchem_info))


/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_67433/2715719528.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top5["PubChem CID"], top5["Drug Name"] = zip(*top5["ligand_smiles"].map(smiles_to_pubchem_info))
/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_67433/2715719528.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top5["PubChem CID"], top5["Drug Name"] = zip(*top5["ligand_smiles"].map(smiles_to_pubchem_info))


In [205]:
top5

,ligand_smiles,protein_sequence,predicted_prob,UniProt ID,Protein Name,PubChem CID,Drug Name,UniProt_ID,Protein_Name
63741,CC(C(=O)NC1=CC=C(C=C1)C(=O)N)OC(=O)COC2=CC=CC=...,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999548,NaN,NaN,45703873,[1-(4-carbamoylanilino)-1-oxopropan-2-yl] 2-(2...,None,None
63184,CC(C(=O)NC1=CC=C(C=C1)F)OC(=O)COC2=CC=CC=C2Cl,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999464,NaN,NaN,78552933,[1-(4-fluoroanilino)-1-oxopropan-2-yl] 2-(2-ch...,None,None
63317,CC(C(=O)NC1=CC=C(C=C1)C(=O)C)OC(=O)COC2=CC=CC=...,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999463,NaN,NaN,45703816,[1-(4-acetylanilino)-1-oxopropan-2-yl] 2-(2-ch...,None,None
63174,CC(C(=O)NC(=O)NC)OC(=O)COC1=CC=CC=C1Cl,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999258,NaN,NaN,45703855,[1-(methylcarbamoylamino)-1-oxopropan-2-yl] 2-...,None,None
63118,CC(C(=O)NC1CCCC2=CC=CC=C12)OC(=O)COC3=CC=CC=C3Cl,MEAKAVARTIRIAPRKVRLVLDLIRGKNAAEAIAILKLTNKASSPV...,0.999235,NaN,NaN,78552935,"[1-oxo-1-(1,2,3,4-tetrahydronaphthalen-1-ylami...",None,None


In [206]:
# keep only columns: protein_sequence, predicted_prob, PubChem CID, Drug Name\
top5_final = top5[["protein_sequence", "predicted_prob", "PubChem CID", "Drug Name"]]

In [208]:
top5_final.to_csv("../data/predicted_top5_dti_interactions.csv", index=False)

In [ ]:
######=========================================================